# Training Split Query Scores for Initial Pool-Only Checkpoints

This notebook tests whether query scores separate clips and when their rankings become stable enough for active learning. The 100% checkpoint is a mature-model ranking reference, not ground truth.

In [ ]:
import json
from itertools import combinations
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
experiment_dir = Path.cwd()
if not (experiment_dir / "query_scores.ipynb").exists():
    experiment_dir = experiment_dir / "experiments" / "1_determine_initial_pool"
runs_dir = experiment_dir / "runs"
train_path = experiment_dir.parents[1] / "src/F3Set/data/f3set-tennis/train.json"
QUERY_BUDGET_PERCENT = 10.0  # Matches active_learning_runs.sh
REFERENCE_POOL_SIZE = 100.0
CANDIDATE_POOL_SIZES = [5.0, 10.0, 12.5, 15.0]
SATURATION_TOLERANCE = 1e-3

with train_path.open() as file:
    train = json.load(file)
frame_by_video = {row["video"]: int(row["num_frames"]) for row in train}
if len(frame_by_video) != len(train):
    raise ValueError("Training video names must be unique")
all_videos = set(frame_by_video)
query_frame_budget = int(np.ceil(sum(frame_by_video.values()) * QUERY_BUDGET_PERCENT / 100))

records, runs, labeled_videos = [], [], {}
score_paths = sorted(runs_dir.glob("*/round_000/train_split_query_scores.json"))
if not score_paths:
    raise FileNotFoundError(f"No train-split query scores found below {runs_dir.resolve()}")
for score_path in score_paths:
    run_dir = score_path.parents[1]
    with (run_dir / "config.json").open() as file:
        config = json.load(file)
    with score_path.open() as file:
        rows = json.load(file)
    with (run_dir / "round_000/labeled_train.json").open() as file:
        labeled = json.load(file)
    pool, seed = float(config["initial_labeled_pool_size"]), int(config["seed"])
    videos = {row["video"] for row in rows}
    if videos != all_videos or len(rows) != len(all_videos):
        raise ValueError(f"{run_dir.name}: score file does not contain the training split exactly once")
    runs.append({"run": run_dir.name, "pool_size": pool, "seed": seed})
    labeled_videos[(pool, seed)] = {row["video"] for row in labeled}
    for row in rows:
        for pooling in ("mean", "max"):
            records.append({"pool_size": pool, "seed": seed, "video": row["video"],
                            "pooling": pooling.upper(), **row[pooling]})
score_df, run_df = pd.DataFrame(records), pd.DataFrame(runs)
metrics = [column for column in score_df if column.endswith(("_measure", "_entropy"))]
metric_labels = dict(zip(metrics, ["Coarse uncertainty", "Fine uncertainty", "Coarse entropy",
                                      "Fine entropy", "Grouped fine entropy"]))
pool_sizes = sorted(score_df.pool_size.unique())
expected = {(pool, seed) for pool in pool_sizes for seed in sorted(run_df.seed.unique())}
available = set(zip(run_df.pool_size, run_df.seed))
print(f"Loaded {len(run_df)} runs, {len(pool_sizes)} pool sizes, and {len(all_videos):,} videos")
print(f"Acquisition budget: {QUERY_BUDGET_PERCENT:g}% = {query_frame_budget:,} frames")
display(run_df.groupby("pool_size").seed.agg(["count", list]))
print("Missing pool/seed score files:", sorted(expected - available) or "none")

## Score distributions and robust dispersion

Violins show all clips across seeds; black points are per-seed medians. The second figure shows the median, IQR, and 5th–95th percentile range. Separate columns prevent MEAN and MAX pooling scales from being conflated.

In [ ]:
colors = dict(zip(pool_sizes, plt.cm.viridis(np.linspace(.05, .95, len(pool_sizes)))))
fig, axes = plt.subplots(len(metrics), 2, figsize=(18, 3.1 * len(metrics)), squeeze=False)
rng = np.random.default_rng(7)
for i, metric in enumerate(metrics):
    for j, pooling in enumerate(("MEAN", "MAX")):
        ax = axes[i, j]
        arrays = [score_df.loc[(score_df.pool_size == pool) & (score_df.pooling == pooling), metric].to_numpy()
                  for pool in pool_sizes]
        violin = ax.violinplot(arrays, positions=range(len(pool_sizes)), widths=.8,
                                showmedians=True, showextrema=False, points=200)
        for body, pool in zip(violin["bodies"], pool_sizes):
            body.set_facecolor(colors[pool]); body.set_edgecolor("none"); body.set_alpha(.55)
        seed_medians = (score_df[score_df.pooling == pooling]
                        .groupby(["pool_size", "seed"])[metric].median().reset_index())
        x = seed_medians.pool_size.map({pool: k for k, pool in enumerate(pool_sizes)}).to_numpy()
        ax.scatter(x + rng.uniform(-.12, .12, len(x)), seed_medians[metric], s=8, c="#202020", alpha=.45)
        ax.set(title=f"{metric_labels[metric]} — {pooling}", xlabel="Initial pool (%)", ylabel="Score")
        ax.set_xticks(range(len(pool_sizes)), [f"{pool:g}" for pool in pool_sizes], rotation=45)
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(len(metrics), 2, figsize=(18, 3.1 * len(metrics)), squeeze=False)
for i, metric in enumerate(metrics):
    for j, pooling in enumerate(("MEAN", "MAX")):
        ax = axes[i, j]
        q = (score_df[score_df.pooling == pooling].groupby("pool_size")[metric]
             .quantile([.05, .25, .5, .75, .95]).unstack())
        x = q.index.to_numpy()
        ax.fill_between(x, q[.25], q[.75], color="#4c78a8", alpha=.22, label="IQR")
        ax.plot(x, q[.5], color="#1f4e79", marker="o", label="Median")
        ax.plot(x, q[.05], color="#4c78a8", ls=":", lw=1, label="5th–95th pct.")
        ax.plot(x, q[.95], color="#4c78a8", ls=":", lw=1)
        ax.set(title=f"{metric_labels[metric]} — {pooling}", xlabel="Initial pool (%)", ylabel="Score")
        if i == j == 0: ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Ranking agreement and acquisition overlap

Spearman correlation uses the common full split. For acquisition overlap, each smaller model and its matched-seed 100% reference rank the smaller run's unlabeled clips, selecting whole clips until the 10% frame budget is met.

In [ ]:
def scores(pool, seed, pooling, metric, candidates=None):
    data = score_df[(score_df.pool_size == pool) & (score_df.seed == seed) & (score_df.pooling == pooling)]
    if candidates is not None: data = data[data.video.isin(candidates)]
    return data.set_index("video")[metric].sort_index()

def select_to_budget(series):
    ordered = (series.rename("score").reset_index()
               .assign(frames=lambda x: x.video.map(frame_by_video))
               .sort_values(["score", "video"], ascending=[False, True], kind="mergesort"))
    if ordered.empty: return set()
    stop = min(np.searchsorted(ordered.frames.cumsum().to_numpy(), query_frame_budget), len(ordered) - 1)
    return set(ordered.iloc[:stop + 1].video)

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else np.nan

reference_rows = []
for pool, seed in sorted(available):
    if (REFERENCE_POOL_SIZE, seed) not in available: continue
    candidates = all_videos - labeled_videos[(pool, seed)]
    for pooling in ("MEAN", "MAX"):
        for metric in metrics:
            current, reference = scores(pool, seed, pooling, metric), scores(100, seed, pooling, metric)
            current_top = select_to_budget(current[current.index.isin(candidates)])
            reference_top = select_to_budget(reference[reference.index.isin(candidates)])
            reference_rows.append({"pool_size": pool, "seed": seed, "pooling": pooling, "metric": metric,
                "strategy": f"{metric_labels[metric]} / {pooling}",
                "reference_rank_rho": current.corr(reference, method="spearman"),
                "reference_top_jaccard": jaccard(current_top, reference_top),
                "reference_top_recall": len(current_top & reference_top) / len(reference_top) if reference_top else np.nan})
reference_df = pd.DataFrame(reference_rows)

def plot_summary(ax, data, value, title):
    for strategy, group in data.groupby("strategy", sort=False):
        summary = group.groupby("pool_size")[value].agg(["mean", "min", "max"]).reset_index()
        ax.plot(summary.pool_size, summary["mean"], marker="o", lw=1.5, label=strategy)
        ax.fill_between(summary.pool_size, summary["min"], summary["max"], alpha=.08)
    ax.set(title=title, xlabel="Initial pool (%)", ylabel=value.replace("_", " ")); ax.grid(alpha=.25)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
smaller = reference_df[reference_df.pool_size < 100]
plot_summary(axes[0], smaller, "reference_rank_rho", "Agreement with matched-seed 100% ranking")
plot_summary(axes[1], smaller, "reference_top_jaccard", "Top-acquisition overlap with 100% reference")
handles, labels = axes[1].get_legend_handles_labels(); fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1, .5), fontsize=8)
fig.tight_layout(rect=(0, 0, .82, 1)); plt.show()

## Cross-seed stability

Rank agreement uses all clips. Top-set agreement uses the intersection of clips unlabeled by both seeds, avoiding a confound from their different random initial pools. Bands span the minimum and maximum seed pair.

In [ ]:
stability_rows = []
for pool in pool_sizes:
    pool_seeds = sorted(run_df.loc[run_df.pool_size == pool, "seed"])
    for left_seed, right_seed in combinations(pool_seeds, 2):
        candidates = all_videos - labeled_videos[(pool, left_seed)] - labeled_videos[(pool, right_seed)]
        for pooling in ("MEAN", "MAX"):
            for metric in metrics:
                left, right = scores(pool, left_seed, pooling, metric), scores(pool, right_seed, pooling, metric)
                left_top = select_to_budget(left[left.index.isin(candidates)])
                right_top = select_to_budget(right[right.index.isin(candidates)])
                stability_rows.append({"pool_size": pool, "seed_pair": f"{left_seed}-{right_seed}",
                    "pooling": pooling, "metric": metric, "strategy": f"{metric_labels[metric]} / {pooling}",
                    "seed_rank_rho": left.corr(right, method="spearman"),
                    "seed_top_jaccard": jaccard(left_top, right_top)})
stability_df = pd.DataFrame(stability_rows)
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
plot_summary(axes[0], stability_df, "seed_rank_rho", "Cross-seed ranking stability")
plot_summary(axes[1], stability_df, "seed_top_jaccard", "Cross-seed top-set stability")
handles, labels = axes[1].get_legend_handles_labels(); fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1, .5), fontsize=8)
fig.tight_layout(rect=(0, 0, .82, 1)); plt.show()

## Strategy redundancy

For each candidate pool, compare the ten metric/pooling combinations using seed-averaged rank correlation and seed-averaged simulated-acquisition overlap.

In [ ]:
strategy_order = [f"{metric_labels[m]} / {p}" for p in ("MEAN", "MAX") for m in metrics]
short_labels = [x.replace(" uncertainty", " UM").replace(" entropy", " EM") for x in strategy_order]
def strategy_matrices(pool):
    correlations, overlaps = [], []
    for seed in sorted(run_df.loc[run_df.pool_size == pool, "seed"]):
        wide = score_df[(score_df.pool_size == pool) & (score_df.seed == seed)].pivot(index="video", columns="pooling", values=metrics)
        wide.columns = [f"{metric_labels[m]} / {p}" for m, p in wide.columns]; wide = wide[strategy_order]
        correlations.append(wide.corr(method="spearman").to_numpy())
        candidates = all_videos - labeled_videos[(pool, seed)]
        chosen = {name: select_to_budget(wide.loc[wide.index.isin(candidates), name]) for name in strategy_order}
        overlaps.append([[jaccard(chosen[a], chosen[b]) for b in strategy_order] for a in strategy_order])
    return np.nanmean(correlations, axis=0), np.nanmean(overlaps, axis=0)

candidate_pools = [pool for pool in CANDIDATE_POOL_SIZES if pool in pool_sizes and pool < 100]
fig, axes = plt.subplots(len(candidate_pools), 2, figsize=(16, max(6, 5.5 * len(candidate_pools))), squeeze=False)
for i, pool in enumerate(candidate_pools):
    for j, (matrix, title) in enumerate(zip(strategy_matrices(pool), ("Rank correlation", "Top-set Jaccard"))):
        ax = axes[i, j]; image = ax.imshow(matrix, vmin=0, vmax=1, cmap="YlGnBu")
        ax.set_title(f"{pool:g}% — mean {title}")
        ax.set_xticks(range(10), short_labels, rotation=60, ha="right", fontsize=7)
        ax.set_yticks(range(10), short_labels, fontsize=7)
        for y in range(10):
            for x in range(10):
                ax.text(x, y, f"{matrix[y, x]:.2f}", ha="center", va="center", fontsize=5,
                        color="white" if matrix[y, x] > .65 else "#202020")
fig.colorbar(image, ax=axes, shrink=.25, label="Agreement"); plt.show()

## Decision table

Dispersion is mean per-seed IQR. `degenerate_fraction` counts scores within 0.001 of 0 or 1. A suggested pool is the smallest whose four agreement measures are within 0.02 of that strategy's best sub-100% value and whose scores are non-degenerate. This transparent plateau heuristic is a diagnostic, not a statistical proof.

In [ ]:
distribution_rows = []
for (pool, seed, pooling), group in score_df.groupby(["pool_size", "seed", "pooling"]):
    for metric in metrics:
        values = group[metric]
        distribution_rows.append({"pool_size": pool, "seed": seed, "pooling": pooling, "metric": metric,
            "strategy": f"{metric_labels[metric]} / {pooling}", "median": values.median(),
            "iqr": values.quantile(.75) - values.quantile(.25),
            "p05_p95_range": values.quantile(.95) - values.quantile(.05),
            "degenerate_fraction": ((values <= SATURATION_TOLERANCE) | (values >= 1-SATURATION_TOLERANCE)).mean()})
distribution_df = pd.DataFrame(distribution_rows)
keys = ["pool_size", "pooling", "metric", "strategy"]
decision = distribution_df.groupby(keys)[["median", "iqr", "p05_p95_range", "degenerate_fraction"]].mean().reset_index()
decision = decision.merge(reference_df.groupby(keys)[["reference_rank_rho", "reference_top_jaccard", "reference_top_recall"]].mean().reset_index(), how="left")
decision = decision.merge(stability_df.groupby(keys)[["seed_rank_rho", "seed_top_jaccard"]].mean().reset_index(), how="left")
agreement = ["reference_rank_rho", "reference_top_jaccard", "seed_rank_rho", "seed_top_jaccard"]
eligible = decision[decision.pool_size < 100].copy()
tests = []
for column in agreement:
    tests.append(eligible[column] >= eligible.groupby("strategy")[column].transform("max") - .02)
eligible["plateau"] = np.logical_and.reduce(tests) & (eligible.iqr > 1e-6) & (eligible.degenerate_fraction < .95)
suggestions = eligible[eligible.plateau].groupby("strategy").pool_size.min().rename("suggested_pool")
decision = decision.merge(suggestions, on="strategy", how="left").sort_values(["strategy", "pool_size"])
decision["suggested"] = decision.pool_size.eq(decision.suggested_pool)
columns = ["pool_size", "strategy", "median", "iqr", "p05_p95_range", *agreement,
           "reference_top_recall", "degenerate_fraction", "suggested"]
display(decision[columns].style.format({c: "{:.3f}" for c in columns if c not in ["strategy", "suggested"]}
        | {"pool_size": "{:.1f}%", "degenerate_fraction": "{:.2%}"}).background_gradient(subset=agreement, cmap="YlGn"))
print("Smallest plateau candidate by strategy:"); display(suggestions.to_frame().style.format("{:.1f}%"))

## Interpretation boundary

These diagnostics do not prove that high-scoring clips improve the trained model. A stronger follow-up should relate score quantiles to per-clip prediction error, then measure realized validation gain after acquisition and retraining.